# 🏦 Session 2 — Resume Training từ Checkpoint Epoch 3

**Yêu cầu:** Đã add dataset `banking-checkpoint` vào notebook này.

Sẽ resume training từ checkpoint-939 (epoch 3/6), train thêm epoch 4-6, evaluate, zip output.

In [ ]:
# === 1) Config ===
GITHUB_USERNAME = "tzin1401"
KAGGLE_USERNAME = "nguynlthduy"
REPO_NAME = "banking-intent-unsloth"
BRANCH = "main"
TRAIN_CONFIG = "configs/train_3b_t4x2.yaml"
CHECKPOINT_DATASET = "banking-checkpoint"

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
print("Repo:", REPO_URL)
print("Config:", TRAIN_CONFIG)

In [ ]:
# === 2) Install dependencies ===
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton -q
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "transformers<=5.5.0" "datasets<4.4.0" "dill>=0.3.9" pyyaml scikit-learn -q
print("\n✅ Dependencies installed")

In [ ]:
# === 3) Clone repo ===
import os

os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_NAME"] = REPO_NAME

if not os.path.exists(REPO_NAME):
    !git clone "$REPO_URL"

%cd "$REPO_NAME"
!git fetch origin
!git checkout "$BRANCH"
!git pull origin "$BRANCH"
print("\n✅ Repo ready")
!git rev-parse --short HEAD

In [ ]:
# === 4) Tìm và copy checkpoint ===
import os, shutil, glob

# Kaggle mount datasets tại nhiều path khác nhau, thử tất cả
possible_roots = [
    f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{CHECKPOINT_DATASET}/{REPO_NAME}",
    f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{CHECKPOINT_DATASET}",
    f"/kaggle/input/{CHECKPOINT_DATASET}/{REPO_NAME}",
    f"/kaggle/input/{CHECKPOINT_DATASET}",
]

# Tìm đường dẫn có thư mục outputs/checkpoint-*
input_base = None
for p in possible_roots:
    if os.path.exists(p) and glob.glob(os.path.join(p, "outputs", "checkpoint-*")):
        input_base = p
        break

# Fallback: tìm tự động
if input_base is None:
    found = glob.glob("/kaggle/input/**/checkpoint-939", recursive=True)
    if found:
        # checkpoint-939 nằm trong outputs/, lấy parent của outputs/
        input_base = os.path.dirname(os.path.dirname(found[0]))

if input_base is None:
    print("❌ Không tìm thấy checkpoint! Cấu trúc /kaggle/input/:")
    for r, d, f in os.walk("/kaggle/input"):
        depth = r.replace("/kaggle/input", "").count(os.sep)
        if depth < 4:
            print("  " * depth + os.path.basename(r) + "/")
    raise FileNotFoundError("Không tìm thấy checkpoint")

print(f"✅ Tìm thấy tại: {input_base}")

# Copy outputs
src_outputs = f"{input_base}/outputs"
if os.path.exists("./outputs"):
    shutil.rmtree("./outputs")
shutil.copytree(src_outputs, "./outputs")

# Copy sample_data
src_data = f"{input_base}/sample_data"
if os.path.exists(src_data):
    if os.path.exists("./sample_data"):
        shutil.rmtree("./sample_data")
    shutil.copytree(src_data, "./sample_data")

# Verify
print("\n📦 Checkpoints:")
!ls -d outputs/checkpoint-*
print("\n📊 Data:")
!wc -l sample_data/train.csv sample_data/test.csv

In [ ]:
# === 5) Bật resume + Train ===
import os, yaml

os.environ["TRAIN_CONFIG"] = TRAIN_CONFIG

with open(TRAIN_CONFIG, "r") as f:
    config = yaml.safe_load(f)

config["resume_from_checkpoint"] = True

with open(TRAIN_CONFIG, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Config: resume_from_checkpoint = True")
print(f"   Total epochs: {config['num_train_epochs']}")
print(f"   Resume từ checkpoint-939 (epoch 3)")
print(f"   Còn train: epoch 4, 5, 6 (~7h)\n")

!python scripts/train.py

In [ ]:
# === 6) Backup ngay sau train ===
import shutil, os

# Copy ra /kaggle/working/ root phòng trường hợp cell sau lỗi
for folder in ["outputs", "sample_data", "configs"]:
    dst = f"/kaggle/working/backup_{folder}"
    if os.path.exists(dst):
        shutil.rmtree(dst)
    if os.path.exists(folder):
        shutil.copytree(folder, dst)

# Show kết quả eval
if os.path.exists("outputs/eval_results.txt"):
    print("=" * 55)
    print(open("outputs/eval_results.txt").read())
    print("=" * 55)
else:
    print("⚠️ Chưa có eval_results.txt")

print("\n✅ Backup xong!")

In [ ]:
# === 7) Package artifacts ===
!python scripts/package_artifacts.py

In [ ]:
# === 8) Zip toàn bộ ===
!zip -r /kaggle/working/banking_final.zip outputs/ sample_data/ configs/ artifacts/ \
    -x '*.git*' '*__pycache__*' '*unsloth_compiled_cache*'

!ls -lh /kaggle/working/banking_final.zip
print("\n✅ XONG! Tải file banking_final.zip từ Output panel bên phải.")